In [1]:
%pip install "graphiti-core[google-genai]"


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# Max concurrent LLM/DB operations (must be set before graphiti_core is imported)
CONCURRENCY_LIMIT = 50
os.environ["SEMAPHORE_LIMIT"] = str(CONCURRENCY_LIMIT)



In [3]:
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from graphiti_core import Graphiti
from datetime import datetime, timezone
from graphiti_core.nodes import EpisodeType
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF

In [4]:
import nest_asyncio, asyncio
nest_asyncio.apply()

In [7]:
from graphiti_core import Graphiti
from graphiti_core.llm_client.gemini_client import GeminiClient, LLMConfig
from graphiti_core.embedder.gemini import GeminiEmbedder, GeminiEmbedderConfig
from graphiti_core.cross_encoder.gemini_reranker_client import GeminiRerankerClient
import graphiti_core.helpers as graphiti_helpers

# Apply concurrency limit (also updates module if graphiti_core was imported earlier)
graphiti_helpers.SEMAPHORE_LIMIT = CONCURRENCY_LIMIT

import os

# Google API key configuration
api_key = os.getenv("GEMINI_API_KEY")

# Initialize Graphiti with Gemini clients
graphiti = Graphiti(
    "bolt://127.0.0.1:7687",
    "neo4j",
    "admin1234",

    llm_client=GeminiClient(
        config=LLMConfig(
            api_key=api_key,
            model="gemini-3.1-pro-preview",      # Best reasoning model
            small_model="gemini-3.6-flash",      # Fast production model
        )
    ),

    embedder=GeminiEmbedder(
        config=GeminiEmbedderConfig(
            api_key=api_key,
            embedding_model="gemini-embedding-001",  # Latest embedding model
            #embedding_dim=1024                     # Verify this matches your wrapper's expected dimension
        )
    ),

    cross_encoder=GeminiRerankerClient(
        config=LLMConfig(
            api_key=api_key,
            model="gemini-3.6-flash",
            small_model="gemini-3.6-flash",
        )
    ),

    max_coroutines=CONCURRENCY_LIMIT,
)

# Now you can use Graphiti with Google Gemini for all components

In [10]:
from graphiti_core import Graphiti
from graphiti_core.utils.maintenance.graph_data_operations import clear_data

await clear_data(graphiti.driver)              # optional for a fresh start
#await graphiti.build_indices_and_constraints() # creates indexes once

In [40]:
async def add_episodes_to_graph(graphiti, episodes, group_id, prefix="Episode"):
    """Add a list of episodes to the graph using Graphiti."""
    print(f"📝 Adding {len(episodes)} episodes to graph...")

    for i, episode in enumerate(episodes):
        name = episode.get('name', f"{prefix} {i+1}")
        content = episode['content']

        # Convert non-string content to JSON
        if not isinstance(content, str):
            content = json.dumps(content)

        # Graphiti method for addin data
        await graphiti.add_episode(
            name=name,
            episode_body=content,
            source=episode['type'],
            source_description=episode['description'],
            reference_time=datetime.now(timezone.utc),
            group_id=group_id
        )

    print(f"✅ Successfully added {len(episodes)} episodes!")


def get_article_from_url(url):
    """Scrape article content from a URL."""
    print(f"📰 Fetching article from: {url}")

    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract date
    date_meta = soup.find("meta", {"name": "DC.date.issued"})
    article_date = date_meta["content"] if date_meta and date_meta.get("content") else "Date not found"

    # Extract article text
    paragraphs = soup.find_all("p")
    filtered = [p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 50]
    article_text = "\n\n".join(filtered).encode("utf-8", "ignore").decode("utf-8")
    article_text = article_text.replace("í", "i")

    print("✅ Article extracted successfully")
    return article_date, article_text


async def search_and_display(graphiti, query, num_results=3,search_filter=None):
    """Search the graph and display results in a clean format."""
    print(f"🔍 Searching for: '{query}'")
    print("-" * 50)

    results = await graphiti.search(query, num_results=num_results,search_filter=search_filter)

    for i, r in enumerate(results, 1):
        print(f"{i}. {r.fact}")
        print(f"   Label: {r.name}")
        print(f"   📅 Valid from: {r.valid_at}")
        if r.invalid_at:
            print(f"   ❌ Invalid at: {r.invalid_at}")
        print()

    return results

async def add_data (episodes):
# Add episodes to the graph
  for episode in episodes:
      if episode[ 'type'] == EpisodeType.json:
          episode['content'] = json. dumps(episode['content' ])
      await graphiti.add_episode(name=episode['name'],
                                 episode_body=episode['content' ],
                                 source=episode['type'],
                                 source_description=episode['description'],
                                 reference_time=datetime.now(timezone.utc),
                                  )
      print(f'Added episode: {episode["name"]}')

In [41]:
episodes = [
    {
        "name": "About Me",
        "content": "Hi, I'm Pradip Nichite. I am the founder and CEO of FutureSmart AI.",
        "type": EpisodeType.text,
        "description": "intro"
    },
    {
        "name": "About FutureSmart AI",
        "content": "FutureSmart AI builds custom AI solutions for clients.",
        "type": EpisodeType.text,
        "description": "company overview"
    }
]

await add_data(episodes)  # helper loops and calls graphiti.add_episode()


Added episode: About Me
Added episode: About FutureSmart AI


In [45]:
# Perform a hybrid search combining semantic similarity and BM25 retrieval
query_1 = "Who Is Founder of FutureSmart AI"
fact_results = await search_and_display(graphiti, query_1,2)
#query_2 = "What solutions does FutureSmart build for clients"
#fact_results = await search_and_display(graphiti, query_2,2)


🔍 Searching for: 'Who Is Founder of FutureSmart AI'
--------------------------------------------------
1. Pradip Nichite is the founder of FutureSmart AI.
   Label: IS_FOUNDER_OF
   📅 Valid from: 2026-07-28 16:17:47.320963+00:00

2. Pradip Nichite is the CEO of FutureSmart AI.
   Label: IS_CEO_OF
   📅 Valid from: 2026-07-28 16:17:47.320963+00:00



In [43]:
from pydantic import BaseModel, Field
from datetime import datetime
from typing import Optional

# Custom Entity Types
class Person(BaseModel):
    """A person entity with biographical information."""
    age: Optional[int] = Field(None, description="Age of the person")
    occupation: Optional[str] = Field(None, description="Current occupation")
    location: Optional[str] = Field(None, description="Current location")
    birth_date: Optional[datetime] = Field(None, description="Date of birth")

class Company(BaseModel):
    """A business organization."""
    industry: Optional[str] = Field(None, description="Primary industry")
    founded_year: Optional[int] = Field(None, description="Year company was founded")
    headquarters: Optional[str] = Field(None, description="Location of headquarters")
    employee_count: Optional[int] = Field(None, description="Number of employees")

class Product(BaseModel):
    """A product or service."""
    category: Optional[str] = Field(None, description="Product category")
    price: Optional[float] = Field(None, description="Price in USD")
    release_date: Optional[datetime] = Field(None, description="Product release date")

# Custom Edge Types
class Employment(BaseModel):
    """Employment relationship between a person and company."""
    position: Optional[str] = Field(None, description="Job title or position")
    start_date: Optional[datetime] = Field(None, description="Employment start date")
    end_date: Optional[datetime] = Field(None, description="Employment end date")
    salary: Optional[float] = Field(None, description="Annual salary in USD")
    is_current: Optional[bool] = Field(None, description="Whether employment is current")

class Investment(BaseModel):
    """Investment relationship between entities."""
    amount: Optional[float] = Field(None, description="Investment amount in USD")
    investment_type: Optional[str] = Field(None, description="Type of investment (equity, debt, etc.)")
    stake_percentage: Optional[float] = Field(None, description="Percentage ownership")
    investment_date: Optional[datetime] = Field(None, description="Date of investment")

class Partnership(BaseModel):
    """Partnership relationship between companies."""
    partnership_type: Optional[str] = Field(None, description="Type of partnership")
    duration: Optional[str] = Field(None, description="Expected duration")
    deal_value: Optional[float] = Field(None, description="Financial value of partnership")


In [44]:
entity_types = {
    "Person": Person,
    "Company": Company,
    "Product": Product
}

edge_types = {
    "Employment": Employment,
    "Investment": Investment,
    "Partnership": Partnership
}

edge_type_map = {
    ("Person", "Company"): ["Employment"],
    ("Company", "Company"): ["Partnership", "Investment"],
    ("Person", "Person"): ["Partnership"],
    ("Entity", "Entity"): ["Investment"],  # Apply to any entity type
}

await graphiti.add_episode(
    name="Business Update",
    episode_body="Sarah joined TechCorp as CTO in January 2023 with a $200K salary. TechCorp partnered with DataCorp in a $5M deal.",
    source_description="Business news",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)


AddEpisodeResults(episode=EpisodicNode(uuid='ada9fed7-8fd5-4818-b7fa-d825753c83c0', name='Business Update', group_id='', labels=[], created_at=datetime.datetime(2026, 7, 28, 16, 18, 57, 213424, tzinfo=datetime.timezone.utc), source=<EpisodeType.message: 'message'>, source_description='Business news', content='Sarah joined TechCorp as CTO in January 2023 with a $200K salary. TechCorp partnered with DataCorp in a $5M deal.', valid_at=datetime.datetime(2026, 7, 28, 21, 48, 57, 213410), entity_edges=['8db10062-d8bf-4022-ae89-4b740bba41dc', '8ff8a29b-ece9-4423-a8c1-182a268e991d'], episode_metadata=None), episodic_edges=[EpisodicEdge(uuid='292f6e8c-daa3-4f6a-a7bc-b9131faeb956', group_id='', source_node_uuid='ada9fed7-8fd5-4818-b7fa-d825753c83c0', target_node_uuid='b25cf8ec-dba4-4af0-86ec-549d2a48e954', created_at=datetime.datetime(2026, 7, 28, 16, 18, 57, 213424, tzinfo=datetime.timezone.utc)), EpisodicEdge(uuid='5d7ee7ea-7017-4925-8101-110337f89108', group_id='', source_node_uuid='ada9fed7-

In [46]:
#You can filter search results to specific entity types or edge types using SearchFilters

from graphiti_core.search.search_filters import SearchFilters

# Search for only specific entity types
search_filter = SearchFilters(
    node_labels=["Person", "Company"]  # Only return Person and Company entities
)
fact_results = await search_and_display(graphiti, "Who works at tech companies?",2,search_filter=search_filter)

🔍 Searching for: 'Who works at tech companies?'
--------------------------------------------------
1. Sarah joined TechCorp as CTO with a $200K salary.
   Label: Employment
   📅 Valid from: 2023-01-01 00:00:00+00:00

2. TechCorp partnered with DataCorp in a $5M deal.
   Label: Partnership
   📅 Valid from: None



In [47]:
await graphiti.add_episode(
    name="Business Update",
    episode_body="Sarah resigned from TechCorp as CTO in May 2024",
    source_description="Business news",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

AddEpisodeResults(episode=EpisodicNode(uuid='bfd9b5af-184c-4ffe-a9ae-abe16e8ec09c', name='Business Update', group_id='', labels=[], created_at=datetime.datetime(2026, 7, 28, 16, 19, 51, 985179, tzinfo=datetime.timezone.utc), source=<EpisodeType.message: 'message'>, source_description='Business news', content='Sarah resigned from TechCorp as CTO in May 2024', valid_at=datetime.datetime(2026, 7, 28, 21, 49, 51, 985165), entity_edges=['da4a3acf-be1f-4156-ada8-112e9a876d4e'], episode_metadata=None), episodic_edges=[EpisodicEdge(uuid='89debbd2-ead2-4ebf-bb10-7ae1a0e117fd', group_id='', source_node_uuid='bfd9b5af-184c-4ffe-a9ae-abe16e8ec09c', target_node_uuid='b25cf8ec-dba4-4af0-86ec-549d2a48e954', created_at=datetime.datetime(2026, 7, 28, 16, 19, 51, 985179, tzinfo=datetime.timezone.utc)), EpisodicEdge(uuid='7f8f7619-9e37-4d1f-8665-f2970c3ea9b8', group_id='', source_node_uuid='bfd9b5af-184c-4ffe-a9ae-abe16e8ec09c', target_node_uuid='00280372-2df9-49f5-b342-43727f435096', created_at=datetime

In [48]:
#You can filter search results to specific entity types or edge types using SearchFilters

from graphiti_core.search.search_filters import SearchFilters

# Search for only specific entity types
search_filter = SearchFilters(
    node_labels=["Person", "Company"]  # Only return Person and Company entities
)
fact_results = await search_and_display(graphiti, "Who works at tech companies?",search_filter=search_filter)
# Search for only specific edge types
search_filter = SearchFilters(
    edge_types=["Employment", "Partnership"]  # Only return Employment and Partnership edges
)
fact_results = await search_and_display(graphiti, "Tell me about business relationships",search_filter=search_filter)

fact_results = await search_and_display(graphiti, "When did Sarah resign from TechCorp?")

🔍 Searching for: 'Who works at tech companies?'
--------------------------------------------------
1. Sarah resigned from her position as CTO at TechCorp.
   Label: Employment
   📅 Valid from: 2023-01-01 00:00:00+00:00
   ❌ Invalid at: 2024-05-01 00:00:00+00:00

2. Sarah joined TechCorp as CTO with a $200K salary.
   Label: Employment
   📅 Valid from: 2023-01-01 00:00:00+00:00

3. TechCorp partnered with DataCorp in a $5M deal.
   Label: Partnership
   📅 Valid from: None

🔍 Searching for: 'Tell me about business relationships'
--------------------------------------------------
1. TechCorp partnered with DataCorp in a $5M deal.
   Label: Partnership
   📅 Valid from: None

2. Sarah resigned from her position as CTO at TechCorp.
   Label: Employment
   📅 Valid from: 2023-01-01 00:00:00+00:00
   ❌ Invalid at: 2024-05-01 00:00:00+00:00

3. Sarah joined TechCorp as CTO with a $200K salary.
   Label: Employment
   📅 Valid from: 2023-01-01 00:00:00+00:00

🔍 Searching for: 'When did Sarah resig

In [49]:
# Scrape a football news article
article_url = "https://www.espn.com/soccer/story/_/id/45783151/marcus-rashford-arrives-barcelona-loan-man-united"
article_date, article_text = get_article_from_url(article_url)

# Create episode from article
espn_episode = {
    'content': f"{article_date}\n\n{article_text}",
    'type': EpisodeType.text,
    'description': "Football transfer news and rumors"
}
print("\n📰 Article Preview:")
print(article_text[:800] + "...\n")
group_id = "la-liga"
# Add article to graph
await add_episodes_to_graph(graphiti, [espn_episode], group_id, prefix="ESPN Transfer News")


📰 Fetching article from: https://www.espn.com/soccer/story/_/id/45783151/marcus-rashford-arrives-barcelona-loan-man-united
✅ Article extracted successfully

📰 Article Preview:
Marcus Rashfordlanded inBarcelonaon Sunday ahead of completing a season-long loan move fromManchester United.

Rashford, 27, will undergo a medical early in the week and, if everything goes to plan, will be presented as a Barça player before the club head off on tour, a source told ESPN.

Barça fly to Asia on Thursday and coach Hansi Flick was keen to have Rashford with the team in Japan and South Korea to give him as much time as possible to bed in before the season starts in August.

- Sources:Rashford close to Barcelona loan move-Nico Williams explains 10-year Athletic extension- Sources:Man Utd close on Mbeumo before U.S. tour

Rashford was cleared to travel to Barcelona after the clubs completed all the necessary paperwork earlier on Sunday.

The terms include an option for the deal t...

📝 Adding 1 episodes

In [50]:
# Query about Barcelona transfer rumors
fact_results = await search_and_display(graphiti, "Who are the players rumored to move to Barcelona?",2)

🔍 Searching for: 'Who are the players rumored to move to Barcelona?'
--------------------------------------------------
1. Barça made a move to sign Nico Williams this summer.
   Label: ATTEMPTED_TO_SIGN
   📅 Valid from: 2025-07-20 20:37:00+00:00

2. Deco told ESPN in May that Barça had made signing a left winger their priority this summer.
   Label: GAVE_STATEMENT_TO
   📅 Valid from: 2025-05-01 00:00:00+00:00



In [51]:
# Simulate new transfer updates

#group id used for multi tenancy 
new_updates = [
    {
        "content": "Lionel Messi is rumored to be transferring to Barcelona from Inter Miami.",
        "type": EpisodeType.message,
        "description": "Latest transfer rumor update"
    },
    {
        "content": "Mark Ogden reports that Marcus Rashford has renewed his contract with Manchester United until 2028 and he is no longer connected to any move or loan to Barcelona anymore.",
        "type": EpisodeType.message,
        "description": "Previous facts updates - update the old facts"
    }
]

# Add updates to graph
await add_episodes_to_graph(graphiti, new_updates, group_id, prefix="Transfer Update")

📝 Adding 2 episodes to graph...
✅ Successfully added 2 episodes!


In [52]:
# Query again about Barcelona transfers - should show updated information
fact_results = await search_and_display(graphiti, "Who are the players rumored to move to Barcelona?",3)

🔍 Searching for: 'Who are the players rumored to move to Barcelona?'
--------------------------------------------------
1. Lionel Messi is rumored to be transferring to Barcelona.
   Label: IS_RUMORED_TO_TRANSFER_TO
   📅 Valid from: 2026-07-28 16:23:37.827624+00:00

2. Lionel Messi is rumored to be transferring from Inter Miami.
   Label: IS_RUMORED_TO_TRANSFER_FROM
   📅 Valid from: 2026-07-28 16:23:37.827624+00:00

3. Marcus Rashford is no longer connected to any move or loan to Barcelona.
   Label: HAS_NO_CONNECTION_TO
   📅 Valid from: 2026-07-28 16:24:09.476587+00:00



In [53]:
fact_results = await search_and_display(graphiti, "What is the latest news about Manchester United?")

🔍 Searching for: 'What is the latest news about Manchester United?'
--------------------------------------------------
1. Manchester United is embarking on a pre-season tour in the United States.
   Label: TO_TOUR
   📅 Valid from: 2025-07-20 20:37:00+00:00

2. Manchester United won a match 2-1 in the Premier League against Manchester City.
   Label: COMPETES_IN
   📅 Valid from: 2025-07-20 20:37:00+00:00

3. Ruben Amorim is a coach at Manchester United.
   Label: COACHES
   📅 Valid from: 2025-07-20 20:37:00+00:00

